<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/3_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. type = bin ()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

def plot_custom_wafer_map(df, select_list, plot_type='bin'):
    """
    df 包含: [x, y, bin, defect_pattern, b64_pic]
    select_list: list of str, 例如 ['27', '38']，這些 bin 會顯示在 legend
    """
    fig = go.Figure()

    # --- 1. 基礎過濾 ---
    exclude_values = ['', '0', 0, '1', 1]
    df_filtered = df[~df['bin'].isin(exclude_values)].copy()

    # 標註是否有照片
    df_filtered['has_pic'] = df_filtered['b64_pic'].apply(
        lambda x: x is not None and str(x).strip() != ''
    )

    # --- 2. 沒照片的點 (純黑背景，無 Legend) ---
    df_no_pic = df_filtered[~df_filtered['has_pic']]
    fig.add_trace(go.Scatter(
        x=df_no_pic['x'], y=df_no_pic['y'],
        mode='markers',
        marker=dict(color='black', size=1),
        name='No Picture',
        showlegend=False, # 隱藏此圖例
        hoverinfo='skip' # 沒照片的點不觸發 hover
    ))

    # --- 3. 有照片的點 (核心邏輯) ---
    if plot_type == 'bin':
        df_with_pic = df_filtered[df_filtered['has_pic']]

        # A. 不在 select_list 內的點：統一顏色，合併為一個 Trace，無 Legend
        df_others = df_with_pic[~df_with_pic['bin'].isin(select_list)]
        if not df_others.empty:
            fig.add_trace(go.Scatter(
                x=df_others['x'], y=df_others['y'],
                mode='markers',
                marker=dict(color='grey', size=7),
                name='Other Bins (with pic)',
                showlegend=False, # 不顯示在 Legend
                customdata=df_others['b64_pic'],
                text=df_others['bin'],
                hovertemplate="<b>Bin: %{text} (Other)</b><br>Coord: (%{x}, %{y})<extra></extra>"
            ))

        # B. 在 select_list 內的點：根據 Bin 區分 Legend 與顏色
        df_focus = df_with_pic[df_with_pic['bin'].isin(select_list)]
        unique_bins = df_focus['bin'].unique()

        # 遍歷每一個獨特的 bin 類別來繪製不同的 Trace
        for bin_val in unique_bins:
            df_bin_group = df_focus[df_focus['bin'] == bin_val]
            fig.add_trace(go.Scatter(
                x=df_bin_group['x'],
                y=df_bin_group['y'],
                mode='markers',
                marker=dict(size=10, line=dict(width=1, color='white')), # 自動分配不同顏色
                name=f'Bin {bin_val}', # 顯示在 Legend
                customdata=df_bin_group['b64_pic'],
                text=df_bin_group['defect_pattern'],
                hovertemplate=(
                    "<b>Target Bin: %{fullData.name}</b><br>" +
                    "Coord: (%{x}, %{y})<br>" +
                    "Pattern: %{text}<extra></extra>"
                )
            ))

    # --- 4. 橢圓邊界 (延續前述功能) ---
    # 使用所有有效點來計算邊界
    if not df_filtered.empty:
        x_min, x_max = df_filtered['x'].min(), df_filtered['x'].max()
        y_min, y_max = df_filtered['y'].min(), df_filtered['y'].max()
        cx, cy = (x_min + x_max)/2, (y_min + y_max)/2
        a, b = (x_max - x_min)/2 * 1.05, (y_max - y_min)/2 * 1.05

        fig.add_shape(
            type="circle", layer='between',
            x0=cx-a, y0=cy-b, x1=cx+a, y1=cy+b,
            line=dict(color="limegreen", width=2)
        )

    # --- 5. 佈局 ---
    fig.update_layout(
        template='plotly_white',
        xaxis=dict(layer='below traces', gridcolor='#f0f0f0'),
        yaxis=dict(layer='below traces', gridcolor='#f0f0f0', scaleanchor="x", scaleratio=1),
        legend=dict(title="Focus Bins", itemsizing='constant'),
        clickmode='event+select'
    )

    return fig

# --- 呼叫範例 ---
# focus_list = ['27', '38', 'A']
# fig = plot_custom_wafer_map(your_df, select_list=focus_list)
# fig.show()

##  2. mode = 'bin' / 'defect_pattern'

In [ ]:
import pandas as pd
import plotly.graph_objects as go

def plot_wafer_map(df, select_list, mode='bin'):
    """
    df 包含: [x, y, bin, defect_pattern, b64_pic]
    select_list: 使用者選定的重點項目清單 (可能是 bin 或 defect_pattern 的值)
    mode: 'bin' 或 'defect_pattern' (決定根據哪個欄位來分組 legend)
    """
    fig = go.Figure()

    # --- 1. 基礎過濾 ---
    # 依照需求：不繪製 bin 為 '', '0', '1' 的點
    exclude_bins = ['', '0', 0, '1', 1]
    df_filtered = df[~df['bin'].isin(exclude_bins)].copy()

    # 判斷是否有照片
    df_filtered['has_pic'] = df_filtered['b64_pic'].apply(
        lambda x: x is not None and str(x).strip() != ''
    )

    # --- 2. 沒照片的點 (純黑背景，無 Legend) ---
    df_no_pic = df_filtered[~df_filtered['has_pic']]
    fig.add_trace(go.Scatter(
        x=df_no_pic['x'], y=df_no_pic['y'],
        mode='markers',
        marker=dict(color='black', size=4, opacity=0.2),
        name='No Picture',
        showlegend=False,
        hoverinfo='skip'
    ))

    # --- 3. 有照片的點分組繪製 ---
    df_with_pic = df_filtered[df_filtered['has_pic']]

    # 根據 mode (bin/defect_pattern) 決定分組欄位 (target_col)
    target_col = mode

    # A. 不在 select_list 內的點：合併為一個灰色 Trace
    df_others = df_with_pic[~df_with_pic[target_col].isin(select_list)]
    if not df_others.empty:
        fig.add_trace(go.Scatter(
            x=df_others['x'], y=df_others['y'],
            mode='markers',
            marker=dict(color='rgba(150, 150, 150, 0.5)', size=7),
            name=f'Other {mode}s',
            showlegend=False,
            customdata=df_others['b64_pic'],
            text=df_others[target_col],
            hovertemplate=f"<b>{mode}: %{{text}} (Other)</b><br>Coord: (%{{x}}, %{{y}})<extra></extra>"
        ))

    # B. 在 select_list 內的重點點：獨立分組 Legend
    df_focus = df_with_pic[df_with_pic[target_col].isin(select_list)]
    unique_targets = df_focus[target_col].unique()

    for val in unique_targets:
        df_sub = df_focus[df_focus[target_col] == val]
        fig.add_trace(go.Scatter(
            x=df_sub['x'], y=df_sub['y'],
            mode='markers',
            marker=dict(size=10, line=dict(width=1, color='white')),
            name=f"{val}", # 這裡直接顯示 Pattern 名稱或 Bin 值
            customdata=df_sub['b64_pic'],
            # 視圖模式顯示另一個欄位的資訊作為補充
            text=df_sub['defect_pattern'] if mode == 'bin' else df_sub['bin'],
            hovertemplate=(
                f"<b>{mode}: {val}</b><br>" +
                "Coord: (%{x}, %{y})<br>" +
                ("Pattern: " if mode == 'bin' else "Bin: ") + "%{text}<extra></extra>"
            )
            # --- 選取點互動樣式 ---
            selected=dict(marker=dict(color='red', opacity=1, size=13)),
            unselected=dict(marker=dict(opacity=0.2))
        ))

    # --- 4. 橢圓邊界 (不遮擋點位，不被網格切到) ---
    if not df_filtered.empty:
        x_min, x_max = df_filtered['x'].min(), df_filtered['x'].max()
        y_min, y_max = df_filtered['y'].min(), df_filtered['y'].max()
        cx, cy = (x_min + x_max)/2, (y_min + y_max)/2
        a, b = (x_max - x_min)/2 * 1.08, (y_max - y_min)/2 * 1.08

        fig.add_shape(
            type="ellipse", layer='between',
            x0=cx-a, y0=cy-b, x1=cx+a, y1=cy+b,
            line=dict(color="limegreen", width=2, dash="dash"),
            fillcolor="rgba(0, 255, 0, 0.02)"
        )

    # --- 5. 佈局設定 ---
    fig.update_layout(
        title=f"Wafer Map Analysis - Mode: {mode}",
        template='plotly_white',
        xaxis=dict(layer='below traces', gridcolor='#eee', zeroline=False),
        yaxis=dict(layer='below traces', gridcolor='#eee', zeroline=False, scaleanchor="x", scaleratio=1),
        legend=dict(title=f"Focus {mode}", traceorder="normal", itemsizing='constant'),
        margin=dict(l=40, r=40, t=60, b=40),
        clickmode='event+select'  # select:未被點擊的點會自動變淡（半透明），被點擊的點則維持原色
    )    # event:僅觸發 plotly_click JavaScript 事件或後端（如 Dash/FigureWidget）的點擊回傳。


    return fig

## 3. callback : 點該點 獲取該點資訊(x,y,...)

In [ ]:
from dash import Dash, html, dcc, Input, Output, callback
import plotly.graph_objects as go

# 假設你已經用之前的函式產出了 fig
# fig = plot_wafer_map(df, select_list, mode='bin')

app = Dash(__name__)

app.layout = html.Div([
    html.H3("Wafer Map 互動監控"),

    # 圖表組件
    dcc.Graph(
        id='wafer-map',
        figure=fig,  # 這裡放入你之前寫好的 fig
        config={'displayModeBar': False} # 隱藏上方工具欄讓畫面更乾淨
    ),

    # 顯示點擊資訊的區塊
    html.Div(id='click-info-display', style={
        'padding': '20px',
        'fontSize': '18px',
        'border': '1px solid #ddd',
        'marginTop': '10px',
        'backgroundColor': '#f9f9f9'
    })
])

# --- Callback 核心邏輯 ---
@callback(
    Output('click-info-display', 'children'),
    Input('wafer-map', 'clickData')
)
def display_click_data(clickData):
    # 如果沒點擊任何東西，顯示提示
    if clickData is None:
        return "請點擊地圖上的彩色重點點以查看資訊"

    # 解析 clickData (Plotly 傳回的字典格式)
    point = clickData['points'][0]
    x_coord = point['x']
    y_coord = point['y']
    bin_val = point['fullData']['name'] # 這是我們之前設定在 Trace name 的 Bin 值

    # 建立回傳內容
    info_list = [
        html.B(f"📍 當前選取位置:"),
        html.P(f"X 座標: {x_coord}"),
        html.P(f"Y 座標: {y_coord}"),
        html.P(f"類別 (Bin/Pattern): {bin_val}")
    ]

    return info_list

if __name__ == '__main__':
    app.run(debug=True)